# Collision experiment — time-series and censored analysis

Run the preparation cells below in order. Session 5.1a then prepares the trial-level table for CSV export.

| Step | Purpose | Main output |
|---|---|---|
| Session 0 | Import libraries and configure the dataset | `analysis_methods`, `thisDataset`, `variable_name` |
| Session 1 | Select animals and locate data folders | `dir_list` |
| Session 5.0 | Define frame-by-frame trial extraction | `sort_raster_locustvr` |
| Session 5.1 | Load selected recordings and extract conditions | `ready_to_plot` |

#### Session 0.0: Set up automatic module reloading
These IPython commands reload imported modules when their source files change.

In [ ]:
%reload_ext autoreload
%autoreload 2

#### Session 0.1: Locate the workflow and import shared helpers
The setup finds the repository root automatically and adds the shared repository modules, this Collision folder, and the sibling `utilities` directory to the import path.

In [ ]:
# Libraries for data selection, processing and plotting
import os,json,sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

current_working_directory = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [current_working_directory, *current_working_directory.parents]
     if (candidate / 'locustvr_extractor.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the UnityDataAnalysis repository root.')
workflow_dir = repo_root / 'analyses' / 'collision'
utilities_dir = repo_root.parent / 'utilities'
for import_dir in (repo_root, workflow_dir, utilities_dir):
    import_path = str(import_dir)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

from sorting_time_series_analysis import follow_behaviour_analysis,generate_points_within_rectangles,calculate_speed,diff_angular_degree,sort_raster_fictrac,plot_velocity_vector_field
from plotting_follow_analysis import *
from time_series_analysis import *
from scipy.stats import circmean
from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt
import seaborn as sns
from useful_tools import select_animals_gpt,find_file,column_name_list,get_fill_between_range,read_seq_config
from data_cleaning import findLongestConseqSubseq,interp_fill

#### Session 0.2: Load settings and select the dataset
Load `analysis_methods_collision.json`, set the mounted data directory in `thisDataset`,
and choose the condition value in `variable_name`.
The commented paths and condition values are examples; only uncommented lines are active.

In [ ]:
json_file = workflow_dir / 'analysis_methods_collision.json'
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())
    
# Set the directory containing the experiment data.
#thisDataset ="D:/MatrexVR_Swarm_Data/RunData"
#thisDataset ="D:/MatrexVR_blackbackground_Data/RunData"
#thisDataset ="D:/MatrexVR_grass1_Data/RunData"
thisDataset = "/Volumes/AG_Couzin-Fuchs/DATA/experiment_trackball_Optomotor/locustVR"
#thisDataset ="D:/MatrexVR_2024_3_Data/RunData"
#thisDataset =r"Z:\DATA\experiment_trackball_Optomotor\locustVR"
#thisDataset =r"C:\Users\neuroLaptop\Documents\MatrexVR_grass1_Data\RunData"
# Condition value used to select animals from the metadata table.
variable_name='black; gregarious'
#variable_name='location'
#variable_name='initial_position'
#variable_name='agent_speed'
# Inspection note: trial 115 in VR1_2024-11-16_155242_score_full may contain a tracking jump.
exp_name=analysis_methods.get("experiment_name")

#### Session 1.0: Select Collision animals and locate data folders
With the current settings, animal metadata are loaded from the LocustVR Google Sheet and rows matching `variable_name` plus the exclusion flag `F` are retained. Session folders must contain a selected animal ID and an HDF5 file.

If table-based selection is disabled, all folders containing HDF5 files are discovered instead.

**Output:** Sorted `dir_list`. This cell does not apply a GN-ID range filter.

In [ ]:
# Configure metadata loading and locate data folders.
# Fail early if the server is disconnected or the dataset path is incorrect.
if not Path(thisDataset).is_dir():
    raise FileNotFoundError(
        f"Dataset directory not found: {thisDataset}. "
        "Connect the server and check thisDataset before selecting folders."
    )

dir_list = []
file_type=".h5"
using_google_sheet=True
exp_name=analysis_methods.get("experiment_name")
if exp_name=='locustvr':
    sheet_name = 'LocustVR'
else:
    sheet_name = "Unity_MatrexVR"

# if type(thisDataset) == str:
#     thisDataset = Path(thisDataset)
if analysis_methods.get("load_individual_data") == True:
    if using_google_sheet==True:
        # database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8"
        #         #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep3I9vBJ2uGgVudb8/edit?usp=sharing
        # url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8"
                #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8/edit?usp=sharing
        url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        #df = pd.read_excel(url, engine='openpyxl')# Alternative for an Excel file rather than a Google Sheet.
        df = pd.read_csv(url)
    else:
        excel_file_path = "Z:/DATA/experiment_trackball_Optomotor/Locusts Management.xlsx"
        print(f"using a database {excel_file_path} from the server but this file might be outdated")
        # Create a 'with' statement to open and read the Excel file
        with pd.ExcelFile(excel_file_path) as xls:
            # Read the Excel sheet into a DataFrame with the sheet name (folder name)
            df = pd.read_excel(xls, sheet_name)
        # Pass each selection condition together with its required value.
    if analysis_methods.get("select_animals_by_condition") == True:
       #animal_of_interest=select_animals_gpt(df,"Independent variable (list up all of them in the experiment)","gregarious_leader_grass","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable (list up all of them in the experiment)","gregarious_leader_black","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","translational_rotational_gain","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","bifuration_constant_speed_constant_distance","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","choice_vr_locust_black_locust","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","choice_vr_locust_sta_black_locust","Background","gray","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","closed_loop_sta_black_locust_open_loop_sta_black_locust","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1",variable_name,"Independent variable2","sta_black_locust_2dir_3_initial_position","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
    else:
        animal_of_interest=df

    if exp_name=='locustvr':
        ID_array=animal_of_interest["ID"].values
        print(f"these are the ID of the animals {ID_array}")
        dir_list = [
        root.replace("\\", "/")
        for root, _, files in os.walk(thisDataset)
        if any(ID in root for ID in ID_array)
        and any(file.endswith(file_type) for file in files)]
    else:
        folder_name=animal_of_interest["folder name"].values
        dir_tile=np.tile(thisDataset, (len(folder_name), 1))
        vr_no=animal_of_interest["VR number"].values
        vr_no = vr_no.astype('int')
        no_food=animal_of_interest["Food retriction (-1 or the number of hours)"].values
        no_food = no_food.astype('int')
        dir_list = [''.join([x[0], '/', y]) for x,y in zip(dir_tile,folder_name)]
        #dir_dict = itertools.zip_longest(dir_list, vr_no.tolist())
        dir_dict = zip(dir_list, vr_no.tolist())
else:
    for root, dirs, files in os.walk(thisDataset):
        for folder in dirs:
            folder_path=os.path.join(root,folder)
            if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                dir_list.append(folder_path.replace("\\", "/"))
dir_list.sort()

#### Session 5.0: Define frame-by-frame trial extraction
`sort_raster_locustvr` reads the focal-animal and trial-summary files, computes walking speed and angular velocity,
and extracts the configured analysis window for each trial.

- Angular velocity uses a Savitzky–Golay derivative with a 50-sample window and polynomial order 3.
- `heading` must be in radians; angular velocity is returned in degrees per second.
- Window offsets are measured relative to the detected trial start; `frame_count` starts at 0 within each extracted window.
- Trials whose requested window extends beyond the recording are skipped.

The function is defined here and called in Session 5.1.

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def sort_raster_locustvr(
    focal_animal_file, summary_file, analysis_methods, count
):
    analysis_window_default = analysis_methods.get("analysis_window")
    analysis_windows_by_speed = analysis_methods.get(
        "analysis_windows_by_speed", {}
    )
    camera_fps = analysis_methods.get("camera_fps")

    df_focal_animal = pd.read_hdf(focal_animal_file)
    df_summary = pd.read_hdf(summary_file)

    split_label = df_summary["trial_label"].str.extract(
        r"Texture(\d+(?:\.\d+)?)_CS(\d+(?:\.\d+)?)_S(\d+(?:\.\d+)?)"
    )
    df_summary["constant_speed"] = split_label[1].astype(float)

    column_list = [
        "labels",
        "elapsed_time",
        "instant_speed",
        "instant_angular_velocity",
    ]

    X = df_focal_animal["X"].to_numpy()
    Y = df_focal_animal["Y"].to_numpy()
    state_type = df_focal_animal["state_type"].to_numpy()
    elapsed_time = df_focal_animal["ts"].to_numpy().astype(float)

    # Fill missing heading values and unwrap angular discontinuities.
    # Assumption: heading is expressed in radians.
    rot_y = df_focal_animal["heading"].to_numpy()
    rot_y = interp_fill(rot_y)
    unwrapped = np.unwrap(rot_y)

    window_length = 50
    polyorder = 3

    # Smoothed angular velocity in radians per second.
    dtheta_dt = savgol_filter(
        unwrapped,
        window_length=window_length,
        polyorder=polyorder,
        deriv=1,
        delta=1.0 / camera_fps,
    )

    # Convert to degrees per second.
    instant_angular_velocity = dtheta_dt * (180.0 / np.pi)

    instant_speed = calculate_speed(
        np.diff(X), np.diff(Y), elapsed_time, 0
    )

    trial_starts = np.where(np.diff(state_type) > 0)[0]

    max_idx = min(
        len(elapsed_time),
        len(instant_speed),
        len(instant_angular_velocity),
    ) - 1

    start_idx_list = []
    labels_list = []
    frame_list = []
    trial_id_list = []

    for i, start in enumerate(trial_starts):
        speed = df_summary.loc[i, "constant_speed"]
        window = analysis_windows_by_speed.get(
            speed, analysis_window_default
        )

        t0, t1 = window
        w_len = int((t1 - t0) * camera_fps)
        s_idx = int(start + t0 * camera_fps)

        if s_idx < 0 or (s_idx + w_len - 1) > max_idx:
            continue

        start_idx_list.append(np.full(w_len, s_idx, dtype=int))
        labels_list.append(
            np.repeat(df_summary.loc[i, "trial_label"], w_len)
        )
        frame_list.append(np.arange(w_len, dtype=int))
        trial_id_list.append(np.repeat(i, w_len))

    if not start_idx_list:
        return pd.DataFrame(
            columns=["animal_id", "frame_count", "trial_id"] + column_list
        )

    start_big = np.concatenate(start_idx_list)
    pts_tile = np.concatenate(frame_list)
    label_tiles = np.concatenate(labels_list)
    trial_id = np.concatenate(trial_id_list)

    indices = start_big + pts_tile

    # Create columns separately to preserve numeric data types.
    dynamics_pd = pd.DataFrame({
        "animal_id": count,
        "frame_count": pts_tile,
        "trial_id": trial_id,
        "labels": label_tiles,
        "elapsed_time": elapsed_time[indices],
        "instant_speed": instant_speed[indices],
        "instant_angular_velocity": instant_angular_velocity[indices],
    })

    return dynamics_pd

#### Session 5.1: Load recordings and prepare frame-level data

##### 1. Review selected data folders
Inspect `dir_list` before loading the recordings.

In [ ]:
dir_list

##### 2. Configure analysis windows and file names
The current windows are −1 to 40 seconds at 1 cm/s and −1 to 10 seconds at 4 cm/s.
`time_series_analysis` determines whether file names use the `_full` suffix.

In [ ]:
time_series_analysis = analysis_methods.get("time_series_analysis")

#analysis_methods.update({"analysis_window": [-1, 10]})

analysis_methods.update({
    "analysis_windows_by_speed": {
        1.0: [-1, 40],
        4.0: [-1, 10],
        
    }
})

file_suffix = "_full" if time_series_analysis else ""
agent_pattern = f"agent{file_suffix}.h5"
xy_pattern = f"XY{file_suffix}.h5"
summary_pattern = f"summary{file_suffix}.h5"


##### 3. Load each selected recording
Find the matching XY and summary files, then extract the trial windows using `sort_raster_locustvr`.
Results are collected in `pd_list`. Animal IDs are assigned from the order of folders in `dir_list`.

In [ ]:
pd_list = []

for index, this_dir in enumerate(dir_list):
    print(f'Processing directory: {this_dir}')
    agent_file = find_file(Path(this_dir), agent_pattern)
    focal_animal_file = find_file(Path(this_dir), xy_pattern)
    summary_file = find_file(Path(this_dir), summary_pattern)
    
    
    this_pd = sort_raster_locustvr(focal_animal_file, summary_file, analysis_methods, index)
    
    pd_list.append(this_pd)


##### 4. Combine frame-level data
Concatenate the per-recording tables into `ready_to_plot`.

In [ ]:
ready_to_plot=pd.concat(pd_list,ignore_index=True)


##### 5. Extract stimulus conditions and inspect the data
Extract texture, stimulus speed and stimulus side from the trial labels.
Texture codes 0 and 1 are displayed as `gregarious` and `black`; other codes retain their values.
The preview below displays up to 100,000 rows.

In [ ]:
split_label = ready_to_plot['labels'].str.extract(r'Texture(\d+(?:\.\d+)?)_CS(\d+(?:\.\d+)?)_S(\d+(?:\.\d+)?)')

ready_to_plot['texture'] = split_label[0].astype(float)
ready_to_plot['constant_speed'] = split_label[1].astype(float)
ready_to_plot['side'] = split_label[2].astype(float)
ready_to_plot.drop(columns=['labels'])
mapping = {0: "gregarious", 1: "black"}
ready_to_plot['texture'] = ready_to_plot['texture'].replace(mapping)
ready_to_plot.head(100000)

#### Session 5.1a: Detect stops and run the censored analysis

**Workflow:** Settings → detect and inspect → export → statistics.
`censored_df` is created directly from `ready_to_plot` with completed response fields.
The detector uses a speed threshold of **0.2** and a sustained-stop duration of **0.5 seconds**.
Times are measured from the search start at −0.5 seconds relative to stimulus onset.

| Stimulus speed | Search window | No-stop observation time |
|---|---|---|
| 1 cm/s | −0.5 to 20 s | 20.5 s |
| 4 cm/s | −0.5 to 5 s | 5.5 s |

`event_observed`: **1 = stop detected**, **0 = no stop within the window**.
`censored_df` contains only valid trials. Invalid recordings and detected stop onsets before stimulus onset
are excluded automatically and listed in `invalid_trials` with their reasons.
A stop at exactly stimulus onset (0 seconds) remains valid.

##### Helper functions — run once
These definitions use raw speed data and the speed-specific recorded analysis window.
Stop onset must be in the search window. Confirmation uses the following 0.5 seconds,
including recorded data after collision when necessary. Animals already stopped at the first
searched sample are excluded as pre-stimulus stops.
The boolean walk-state branch of the supplied detector is not used here.

In [ ]:
def detect_stop_events(arr, analysis_methods,response_window=[-0.5,4],sustained_time_for_behaviour=0.5,save_path=Path.cwd()):
    analysis_window=analysis_methods.get("analysis_window")
    camera_fps=analysis_methods.get("camera_fps")
    plotting_tbt_overview=analysis_methods.get("plotting_tbt_overview")
    n_trials = arr.shape[0]
    event_detected = np.zeros(n_trials, dtype=bool)
    event_index = np.full(n_trials, -1, dtype=int)
    event_time = np.full(n_trials, np.nan)
    # Define frame number for analysis windows
    relative_time = analysis_window[0] + np.arange(arr.shape[1]) / camera_fps
    sustain_samples = int(sustained_time_for_behaviour*camera_fps)
    analysis_mask = (relative_time > response_window[0]) & (relative_time <= response_window[1])
    analysis_indices = np.where(analysis_mask)[0]
    if arr.dtype == 'float64':## feed raw velocity data. Otherwises, use boolean value of walk/stop states
        varying_deceleration_threshold=False
        fixed_threshold=0.2#0.2 is an arbitary threshold, which should be adjusted based on the setup.walk_thresholds=[0.2,0.5]
        if varying_deceleration_threshold:
            baseline_mask = (relative_time >= -1.5) & (relative_time < response_window[0])
            baseline_data = arr[:, baseline_mask]
            baseline_mean = baseline_data.mean(axis=1)
            baseline_std2 = 2 * baseline_data.std(axis=1, ddof=1)
            threshold = baseline_mean - baseline_std2
        else:
            threshold = fixed_threshold*np.ones(n_trials)
        # ------------------------------------------------------------------
        # Detect sustained threshold crossing
        # ------------------------------------------------------------------
        for trial in range(n_trials):

            th = threshold[trial]
            trace = arr[trial]
            if plotting_tbt_overview:
                plt.figure(figsize=(8, 6))
                plt.plot(np.arange(trace.shape[0]),trace, linewidth=0.2)
                plt.savefig(save_path / f'velocity_trial{trial}.png')
                plt.close()
            for idx in analysis_indices:

                # Not enough samples left for confirmation
                if idx + sustain_samples >= len(trace):
                    break

                if trace[idx] < th:

                    # Require all following samples (0.5 sec) remain below threshold
                    if np.all(trace[idx:idx + sustain_samples + 1] < th):
                        event_detected[trial] = True
                        event_index[trial] = idx
                        event_time[trial] = relative_time[idx]
                        if plotting_tbt_overview:
                            plt.figure(figsize=(8, 6))
                            plt.plot(np.arange(trace.shape[0]),trace, linewidth=0.2)
                            plt.plot(idx,trace[idx],'o',color='r')
                            plt.savefig(save_path / f'velocity_trial{trial}.png')
                            plt.close()
                        break
    else:
        walk_state=arr[:,analysis_indices]
        (id,frame_number)=np.where(np.diff(walk_state))
        event_index[id]=frame_number
        if plotting_tbt_overview:
            for trial in range(n_trials):
                trace = arr[trial]
                plt.figure(figsize=(8, 6))
                plt.plot(np.arange(trace.shape[0]),trace, linewidth=0.2)
                if trial in id:
                    plt.plot(event_index[trial]+int(camera_fps*(abs(analysis_window[0])+response_window[0])),0.5,'o',color='r')
                plt.savefig(save_path / f'walk_state_trial{trial}.png')
                plt.close()
    return event_index


In [ ]:
def build_censored_stop_table(
    frame_data, analysis_methods, response_windows_by_speed,
    sustained_stop_seconds=0.5,
):
    """Apply detect_stop_events to raw speeds, separately for each animal/trial.

    Times are seconds since the start of the speed-specific response window.
    Stop onset must lie inside (window_start, window_end]. The full confirmation
    period may extend beyond window_end, using the recorded post-window data.
    Invalid recordings and stops before stimulus onset are excluded from the
    analysis table and retained with a status reason in the diagnostic table.
    """
    required = [
        "animal_id", "trial_id", "texture", "constant_speed",
        "frame_count", "instant_speed",
    ]
    data = frame_data[required].copy()
    for col in ["constant_speed", "frame_count", "instant_speed"]:
        data[col] = pd.to_numeric(data[col], errors="raise")
    if data.empty or data[required[:-1]].isna().any().any():
        raise ValueError("Trial identifiers, conditions and frame counts must be present.")
    fps = float(analysis_methods["camera_fps"])
    if not np.isfinite(fps) or fps <= 0 or sustained_stop_seconds <= 0:
        raise ValueError("Frame rate and sustained-stop duration must be positive.")
    sustain_frames = int(sustained_stop_seconds * fps)
    if sustain_frames < 1:
        raise ValueError("The sustained-stop interval must contain at least one frame.")

    rows = []
    for (animal_id, trial_id), group in data.groupby(
        ["animal_id", "trial_id"], sort=True
    ):
        group = group.sort_values("frame_count")
        if group[["texture", "constant_speed"]].nunique().gt(1).any():
            raise ValueError(f"Multiple conditions in trial {(animal_id, trial_id)}.")
        speed = float(group["constant_speed"].iloc[0])
        if speed not in response_windows_by_speed:
            raise ValueError(f"No response window configured for speed {speed}.")
        response_start, response_end = response_windows_by_speed[speed]
        if not np.isfinite([response_start, response_end]).all() or response_end <= response_start:
            raise ValueError(f"Invalid response window for speed {speed}.")
        windows = analysis_methods.get("analysis_windows_by_speed", {})
        trial_window = windows.get(speed, windows.get(str(speed), analysis_methods.get("analysis_window")))
        if trial_window is None:
            raise ValueError(f"No recorded analysis window configured for speed {speed}.")
        t0, t1 = trial_window
        trace = group["instant_speed"].to_numpy(dtype=np.float64)
        times = float(t0) + np.arange(len(trace)) / fps
        searched = np.flatnonzero((times > response_start) & (times <= response_end))
        row = {
            "locust_id": animal_id, "trial_order": trial_id,
            "colour": group["texture"].iloc[0], "speed": speed,
            "slowing_time": np.nan, "event_observed": np.nan,
            "response_start_s": response_start, "response_end_s": response_end,
            "stop_frame": np.nan, "stop_time_from_stimulus_s": np.nan,
            "status": "valid",
        }
        full_frames = np.array_equal(group["frame_count"].to_numpy(), np.arange(len(trace)))
        if not full_frames or not np.isclose(len(trace), (t1 - t0) * fps):
            row["status"] = "incomplete_or_duplicate_frames"
        elif not len(searched) or t0 > response_start or times[-1] < response_end:
            row["status"] = "response_window_not_recorded"
        elif searched[-1] + sustain_frames >= len(trace):
            row["status"] = "insufficient_confirmation_data"
        elif not np.isfinite(trace[searched[0]:searched[-1] + sustain_frames + 1]).all():
            row["status"] = "missing_or_nonfinite_speed"
        else:
            settings = dict(analysis_methods)
            settings["analysis_window"] = list(trial_window)
            settings["plotting_tbt_overview"] = False
            event_index = int(detect_stop_events(
                trace[None, :], settings,
                response_window=(response_start, response_end),
                sustained_time_for_behaviour=sustained_stop_seconds,
            )[0])
            if event_index >= 0:
                row["stop_frame"] = event_index
                row["stop_time_from_stimulus_s"] = times[event_index]
                if times[event_index] < 0:
                    row["status"] = "stop_before_stimulus_onset"
                else:
                    row["event_observed"] = 1
                    row["slowing_time"] = times[event_index] - response_start
            else:
                row["event_observed"] = 0
                row["slowing_time"] = response_end - response_start
        rows.append(row)

    details = pd.DataFrame(rows)
    columns = ["locust_id", "trial_order", "colour", "speed", "slowing_time", "event_observed"]
    valid_trials = details.loc[details["status"].eq("valid"), columns].copy()
    valid_trials["event_observed"] = valid_trials["event_observed"].astype(int)
    return valid_trials.reset_index(drop=True), details

##### 1. Settings
Adjust collision times and the sustained-stop duration here. Generated CSV files and model results are written below this workflow's `results/` directory.

In [ ]:
# Stop onset is searched from 0.5 seconds before stimulus onset to collision.
# Collision times follow the existing frames_to_focal values at 100 fps:
# 2000 frames -> 20 s at 1 cm/s; 500 frames -> 5 s at 4 cm/s.
collision_time_by_speed = {1.0: 20.0, 4.0: 5.0}
response_windows_by_speed = {
    speed: (-0.5, collision_time)
    for speed, collision_time in collision_time_by_speed.items()
}
sustained_stop_seconds = 0.5

# Keep generated tables and model results inside this workflow's results directory.
results_dir = workflow_dir / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
csv_path = results_dir / 'out_censored.csv'


##### 2. Detect stops and inspect the table
Creates `censored_df` directly, with one row per valid animal/trial.
Trials with invalid measurements or a stop onset before stimulus onset are excluded automatically.
`invalid_trials` lists excluded trials and reasons; `stop_event_details` retains all diagnostic records.
`slowing_time` remains measured from the search start at −0.5 seconds.

In [ ]:
# Build the analysis table from valid trials only.
censored_df, stop_event_details = build_censored_stop_table(
    ready_to_plot,
    analysis_methods,
    response_windows_by_speed,
    sustained_stop_seconds=sustained_stop_seconds,
)
invalid_trials = stop_event_details.loc[
    stop_event_details["status"].ne("valid")
].reset_index(drop=True)

print(f"Valid trials: {len(censored_df)} | Excluded trials: {len(invalid_trials)}")
print(stop_event_details["status"].value_counts().to_string())
censored_df.head(20)

##### 3. Export valid trials and the exclusion log
The main CSV contains only valid trials and is used by the statistics script.
A separate `_invalid_trials.csv` records excluded trials and reasons.
Both files are overwritten when this cell is run.

In [ ]:
if censored_df.empty:
    raise ValueError("No valid trials remain for export and analysis.")

censored_df.to_csv(csv_path, index=False)
invalid_trials_path = csv_path.with_name(f"{csv_path.stem}_invalid_trials.csv")
invalid_trials.to_csv(invalid_trials_path, index=False)
print(f"Saved {len(censored_df)} valid trials to: {csv_path}")
print(f"Saved {len(invalid_trials)} excluded trials to: {invalid_trials_path}")

##### 4. Run the censored repeated-measures statistics
Reads `results/out_censored.csv` and calls `censored_statistics.py`. Model summaries, comparisons and figures are saved in `results/locust_censored_results/`. Sampling may take considerable time.

In [ ]:
from censored_statistics import main

results = main(
    csv_path,
    output_dir=csv_path.parent / "locust_censored_results",
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
)

#### Session 5.2: Prepare walking distance and heading change
`calculate_cumulative_walking_distance` adds cumulative walking distance and heading change to the frame data.
The function is then applied to `ready_to_plot`, with the result stored in `ready_to_plot1`.
The existing default is 100 frames per second.

##### Define the function

In [ ]:
def calculate_cumulative_walking_distance(df, fps=100):
    df = df.copy()

    df['instant_speed'] = pd.to_numeric(df['instant_speed'], errors='coerce')
    df['instant_angular_velocity'] = pd.to_numeric(df['instant_angular_velocity'], errors='coerce')


 
    df = df.sort_values(by=['animal_id', 'trial_id', 'frame_count'])

    dt = 1.0 / fps

    
    df['distance_per_frame'] = df['instant_speed'] * dt

    
    df['cumulative_distance'] = df.groupby(['animal_id', 'trial_id'])['distance_per_frame'].cumsum()

    df['heading_diff_per_frame'] = df['instant_angular_velocity'] * dt

    
    df['cumulative_heading'] = df.groupby(['animal_id', 'trial_id'])['heading_diff_per_frame'].cumsum()


    return df


##### Apply the function and inspect the result

In [ ]:
ready_to_plot1 = calculate_cumulative_walking_distance(ready_to_plot)
ready_to_plot1.head(20)

#### Session 5.3: Existing deceleration detectors (optional)
These analyses are independent of `censored_df` and produce separate result tables.
Analysis windows and thresholds retain their existing definitions.

##### Method A: First crossing below the baseline threshold
Baseline before frame 100; threshold = mean minus three standard deviations.
Output: `deceleration_df`.

In [ ]:
def detect_first_deceleration_frame(df, frame_threshold=100):
    """
    Detect the first frame where instant_speed falls below
    mean(baseline) - 3*std(baseline) after frame_threshold.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing at least:
        animal_id, trial_id, frame_count, instant_speed,
        constant_speed, texture
    frame_threshold : int, default=100
        Frames before this threshold are used as baseline.
        Detection starts at this frame.

    Returns
    -------
    pandas.DataFrame
        Summary dataframe with one row per trial.
    """
    import numpy as np
    import pandas as pd

    results = []

    for key, group in df.groupby(["animal_id", "trial_id"]):
        group = group.sort_values("frame_count").copy()

        animal_id, trial_id = key

        # Additional information
        constant_speed = group["constant_speed"].iloc[0]
        texture = group["texture"].iloc[0]

        # End frame depends on constant_speed
        if constant_speed == 1:
            end_point = 2300
        elif constant_speed == 4:
            end_point = 650
        else:
            end_point = np.nan

        # Frames until the stimulus reaches the focal animal
        if constant_speed == 1:
            frames_to_focal = 2000
        elif constant_speed == 4:
            frames_to_focal = 500
        else:
            frames_to_focal = np.nan

        # Baseline
        speed_first100 = group.loc[group["frame_count"] < frame_threshold, "instant_speed"]
        mean_speed = speed_first100.mean()
        std_speed = speed_first100.std()
        threshold = mean_speed - 3 * std_speed

        # Transition: first crossing below the threshold
        below = group["instant_speed"] < threshold
        transition = below & ~below.shift(1, fill_value=False)

        # Restrict detection to the valid interval
        valid_transition = (
            transition
            & (group["frame_count"] >= frame_threshold)
            & (group["frame_count"] < end_point)
        )

        # First event or NaN
        if valid_transition.any():
            first_frame = group.loc[valid_transition, "frame_count"].iloc[0]
        else:
            first_frame = np.nan

        results.append({
            "animal_id": animal_id,
            "trial_id": trial_id,
            "texture": texture,
            "constant_speed": constant_speed,
            "frame_threshold": frame_threshold,
            "frames_to_focal": frames_to_focal,
            "mean_instant_speed_first100": mean_speed,
            "std_instant_speed_first100": std_speed,
            "deceleration_threshold": threshold,
            "first_deceleration_frame": first_frame,
            "end_point": end_point
        })

    return pd.DataFrame(results)

In [ ]:
deceleration_df = detect_first_deceleration_frame(ready_to_plot1)
deceleration_df

##### Method B: Threshold crossing with a locally decreasing speed
This method additionally requires a locally negative speed trend.
Detection starts at frame 400 (1 cm/s) or 175 (4 cm/s).
Output: `scoond_deceleration_df` (existing variable name).

In [ ]:
def detect_second_deceleration_event(df):
    """
    Detect the first valid deceleration event in a speed-dependent analysis window,
    using a locally smoothed negative speed trend at the transition point.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing at least:
        animal_id, trial_id, frame_count, instant_speed,
        constant_speed, texture

    Returns
    -------
    pandas.DataFrame
        One row per trial with summary information and the detected
        first valid deceleration frame.
    """
    import numpy as np
    import pandas as pd

    results = []

    for key, group in df.groupby(["animal_id", "trial_id"]):
        group = group.sort_values("frame_count").copy()

        animal_id, trial_id = key

        # Additional trial information
        constant_speed = group["constant_speed"].iloc[0]
        texture = group["texture"].iloc[0]

        # frame_threshold depends on constant_speed
        if constant_speed == 1:
            frame_threshold = 400   # 100 (pre-choice) + 300 (3 cm distance)
        elif constant_speed == 4:
            frame_threshold = 175   # 100 (pre-choice) + 75 (3 cm distance)
        else:
            frame_threshold = np.nan

        # End frame depends on constant_speed
        if constant_speed == 1:
            end_point = 2300
        elif constant_speed == 4:
            end_point = 650
        else:
            end_point = np.nan

        # Frames until the stimulus reaches the focal animal
        if constant_speed == 1:
            frames_to_focal = 2000
        elif constant_speed == 4:
            frames_to_focal = 500
        else:
            frames_to_focal = np.nan

        # Baseline speed before the analysis interval
        speed_first100 = group.loc[group["frame_count"] < frame_threshold, "instant_speed"]
        mean_speed = speed_first100.mean()
        std_speed = speed_first100.std()
        threshold = mean_speed - 3 * std_speed

        # Threshold-Crossing
        below = group["instant_speed"] < threshold
        transition = below & ~below.shift(1, fill_value=False)

        # Robust local deceleration
        rolling_speed = group["instant_speed"].rolling(
            window=5,
            center=True,
            min_periods=1
        ).mean()

        rolling_diff = rolling_speed.diff()

        negative_local = rolling_diff.rolling(
            window=3,
            center=True,
            min_periods=1
        ).mean() < 0

        # Valid transition
        valid_transition = (
            transition
            & (group["frame_count"] >= frame_threshold)
            & (group["frame_count"] < end_point)
            & negative_local
        )

        # First valid event or NaN
        if valid_transition.any():
            first_frame = group.loc[valid_transition, "frame_count"].iloc[0]
        else:
            first_frame = np.nan

        results.append({
            "animal_id": animal_id,
            "trial_id": trial_id,
            "texture": texture,
            "constant_speed": constant_speed,
            "frame_threshold": frame_threshold,
            "frames_to_focal": frames_to_focal,
            "mean_instant_speed_first100": mean_speed,
            "std_instant_speed_first100": std_speed,
            "deceleration_threshold": threshold,
            "first_deceleration_frame": first_frame,
            "end_point": end_point
        })

    return pd.DataFrame(results)

In [ ]:
scoond_deceleration_df = detect_second_deceleration_event(ready_to_plot1)
scoond_deceleration_df

#### Session 5.4: Plot deceleration timing
The following plots use `scoond_deceleration_df` from Method B.

##### Individual trials as a stripplot

In [ ]:

def plot_deceleration_stripplot(df):
    """
    Create a stripplot of first deceleration frames grouped by
    texture and constant_speed.

    Parameters
    ----------
    df : pandas.DataFrame
        Output dataframe from deceleration detection function.
        Must contain:
        texture, constant_speed, first_deceleration_frame
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Category for the x-axis
    df = df.copy()
    df["group"] = (
        df["texture"].astype(str)
        + "_CS" +
        df["constant_speed"].astype(str)
    )

    plt.figure()

    sns.stripplot(
        data=df,
        x="group",
        y="first_deceleration_frame",
        jitter=True
    )

    plt.ylim(bottom=0)
    plt.xticks(rotation=45)
    plt.ylabel("First Deceleration Frame")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_deceleration_stripplot(scoond_deceleration_df)

##### Distribution as a violin plot with individual observations

In [ ]:
def plot_deceleration_violin(df):
    """
    Violin plot and stripplot of the first deceleration frame,
    grouped by texture and constant_speed.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    df = df.copy()

    # Category for the x-axis
    df["group"] = (
        df["texture"].astype(str)
        + "_CS" +
        df["constant_speed"].astype(str)
    )

    # Optional: remove NaNs, which cannot be plotted
    df = df.dropna(subset=["first_deceleration_frame"])

    plt.figure(figsize=(10, 6))

    sns.violinplot(
        data=df,
        x="group",
        y="first_deceleration_frame",
        inner="box",
        cut=0,
        palette="muted",
        linewidth=2
    )

    sns.stripplot(
        data=df,
        x="group",
        y="first_deceleration_frame",
        color="black",
        size=3,
        alpha=0.3,
        jitter=True
    )

    plt.ylim(bottom=0)
    plt.xticks(rotation=45)
    plt.ylabel("First Deceleration Frame")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_deceleration_violin(scoond_deceleration_df)

#### Session 5.5: Calculate and plot distance to collision
The conversion adds `frames_to_collision` and `deceleration_dis` to `scoond_deceleration_df`.
The plotting function below requires a table containing `deceleration_dis`.
The existing plot call uses `deceleration_df` from Method A;
that table must also contain the distance column before plotting.

##### Calculate distance

In [ ]:
def deceleration_distance_to_collision(df, fps=100):
    df = df.copy()

    # frames_to_collision depends on constant_speed
    df["frames_to_collision"] = np.where(
        df["constant_speed"] == 1,
        df["frames_to_focal"] - (df["first_deceleration_frame"] - 100),
        np.where(
            df["constant_speed"] == 4,
            df["frames_to_focal"] - (df["first_deceleration_frame"] - 100),
            np.nan
        )
    )

    # Calculate distance
    df["deceleration_dis"] = (
        df["frames_to_collision"] / fps
    ) * df["constant_speed"]

    return df

In [ ]:
scoond_deceleration_df = deceleration_distance_to_collision(scoond_deceleration_df)
scoond_deceleration_df

##### Plot distance

In [ ]:
def plot_deceleration_distance(df):
    import matplotlib.pyplot as plt
    import seaborn as sns

    df = df.copy()

    df["group"] = (
        df["texture"].astype(str)
        + "_CS" +
        df["constant_speed"].astype(str)
    )

    plt.figure(figsize=(10, 6))

    sns.violinplot(
        data=df,
        x="group",
        y="deceleration_dis",
        inner="box",
        cut=0,
        palette="muted",
        linewidth=2
    )

    sns.stripplot(
        data=df,
        x="group",
        y="deceleration_dis",
        color="black",
        size=3,
        alpha=0.3,
        jitter=True
    )

    plt.ylim(bottom=0)
    plt.xticks(rotation=45)
    plt.ylabel("deceleration_dis")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_deceleration_distance(deceleration_df)

#### Session 5.6: Existing ANOVA and Tukey analyses (optional)
These cells analyse `deceleration_dis` from `scoond_deceleration_df`.
The separate script `censored_statistics.py` is not called here.

##### Two-way ANOVA

In [ ]:
def two_way_anova_deceleration(df):
    """
    Two-way ANOVA:
    Factor 1: texture
    Factor 2: constant_speed
    Response: deceleration_dis
    """
    import pandas as pd
    import statsmodels.api as sm
    from statsmodels.formula.api import ols

    df = df.copy()

    # Remove NaNs before ANOVA
    df = df.dropna(subset=["deceleration_dis"])

    # Treat factors as categorical variables
    df["texture"] = df["texture"].astype("category")
    df["constant_speed"] = df["constant_speed"].astype("category")

    # Linear model with interaction
    model = ols(
        "deceleration_dis ~ C(texture) * C(constant_speed)",
        data=df
    ).fit()

    # Two-way ANOVA
    anova_table = sm.stats.anova_lm(model, typ=2)

    return anova_table

In [ ]:
anova_results = two_way_anova_deceleration(scoond_deceleration_df)
print(anova_results)

##### Tukey post-hoc comparisons

In [ ]:
def tukey_posthoc_deceleration(df):
    """
    Tukey HSD post-hoc test for deceleration_dis
    across groups (texture × constant_speed).
    """
    import pandas as pd
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    df = df.copy()

    # Remove NaNs
    df = df.dropna(subset=["deceleration_dis"])

    # Grouping variable matching the plot
    df["group"] = (
        df["texture"].astype(str)
        + "_CS" +
        df["constant_speed"].astype(str)
    )

    # Tukey HSD
    tukey = pairwise_tukeyhsd(
        endog=df["deceleration_dis"],
        groups=df["group"],
        alpha=0.05
    )

    return tukey

In [ ]:
tukey_results = tukey_posthoc_deceleration(scoond_deceleration_df)
print(tukey_results)

#### Session 5.7: Inspect time courses
The following plots use frame data from `ready_to_plot1` or `ready_to_plot2`.

##### 5.7.1: Cumulative walking distance per trial

In [ ]:
def walking_distance_over_time(df):
    df = df.copy()

    textures = sorted(df['texture'].unique())
    speeds   = sorted(df['constant_speed'].unique())

    fig, ax = plt.subplots(len(speeds), len(textures),
                           figsize=(6 * len(textures), 4 * len(speeds)),
                           sharey='row')

    grouped = df.groupby(['constant_speed', 'texture', 'animal_id', 'trial_id'])

    for (speed, text, animal_id, trial_id), group in grouped:
        i = speeds.index(speed)
        j = textures.index(text)

        ax[i, j].plot(group['frame_count'],
                      group['cumulative_distance'],
                      alpha=0.3)

        ax[i, j].axvline(100, color='black', linestyle='--', linewidth=1)
        

   
    for i, speed in enumerate(speeds):
        if speed == 1.0:
            y_max = 300
            
            collision_line = 1900
        elif speed == 4.0:
            y_max = 100
            
            collision_line = 550
        else:
            y_max = 100
            

        for j, text in enumerate(textures):
            ax[i, j].set_ylim(0, y_max)
            ax[i, j].set_title(f"{speed} cm/s – {text}")
        
            ax[i, j].axvline(collision_line, color='red', linestyle='--', linewidth=1)

    plt.tight_layout()
    return fig, ax


In [ ]:
walking_distance_over_time(ready_to_plot1)

##### 5.7.2: Filter trials by distance travelled
The example selects trials with at least 10 distance units travelled between 1 and 5 seconds.
Output: `ready_to_plot2`.

In [ ]:
def filter_trials_by_distance_with_framecount(df, time_window_s=(0, 10), min_distance=10.0, fps=100):
    df = df.copy()

    frame_start = int(time_window_s[0] * fps)
    frame_end   = int(time_window_s[1] * fps)

    
    in_window = (df["frame_count"] >= frame_start) & (df["frame_count"] <= frame_end)

    
    dist_in_window = (
        df.loc[in_window]
          .groupby(["animal_id", "trial_id"])["cumulative_distance"]
          .agg(["min", "max"])
    )
    dist_in_window["distance_in_window"] = dist_in_window["max"] - dist_in_window["min"]
    dist_in_window = dist_in_window[["distance_in_window"]].reset_index()

    
    df = df.merge(dist_in_window, on=["animal_id", "trial_id"], how="left")
    df["distance_in_window"] = df["distance_in_window"].fillna(0)

   
    df = df[df["distance_in_window"] >= min_distance]

    return df


In [ ]:
ready_to_plot2 = filter_trials_by_distance_with_framecount(ready_to_plot1, time_window_s=(1, 5), min_distance=10.0)
walking_distance_over_time(ready_to_plot2)

##### 5.7.3: Speed per trial
The existing call plots all trials from `ready_to_plot1`.
`trail_from_one_individual` is an additional selection for animal 21 that is not currently used by the plot call.

In [ ]:
def speed_over_time(df):
    df = df.copy()

    textures = sorted(df['texture'].unique())
    speeds   = sorted(df['constant_speed'].unique())

    fig, ax = plt.subplots(len(speeds), len(textures),
                           figsize=(6 * len(textures), 4 * len(speeds)),
                           sharey='row')

    grouped = df.groupby(['constant_speed', 'texture', 'animal_id', 'trial_id'])

    for (speed, text, animal_id, trial_id), group in grouped:
        i = speeds.index(speed)
        j = textures.index(text)

        ax[i, j].plot(group['frame_count'],
                      group['instant_speed'],
                      alpha=0.3)

        ax[i, j].axvline(100, color='black', linestyle='--', linewidth=1)
        


    for i, speed in enumerate(speeds):
        if speed == 1.0:
            y_max = 15     
            collision_line = 1900
            reapearance_line = 2300
        elif speed == 4.0:
            y_max = 15
            collision_line = 550
            reapearance_line = 650     
        else:
            y_max = 10

        for j, text in enumerate(textures):
            ax[i, j].set_ylim(0, y_max)
            ax[i, j].set_title(f"{speed} cm/s – {text}")
            ax[i, j].axvline(collision_line, color='red', linestyle='--', linewidth=1)
            ax[i, j].axvline(reapearance_line, color='green', linestyle='--', linewidth=1)

    plt.tight_layout()
    return fig, ax



In [ ]:
trail_from_one_individual = ready_to_plot1[
    (ready_to_plot1["animal_id"] == 21)]
speed_over_time(ready_to_plot1)

##### 5.7.4: Mean speed per animal
Trials are averaged within each animal, condition and frame.
The two inspection cells show the most recently used measurement file and `ready_to_plot1`.

In [ ]:

def speed_over_time_per_ID(df):
    df = df.copy()

    textures = sorted(df['texture'].unique())
    speeds   = sorted(df['constant_speed'].unique())

    fig, ax = plt.subplots(
        len(speeds),
        len(textures),
        figsize=(6 * len(textures), 4 * len(speeds)),
        sharey='row'
    )

    df_mean = (
        df
        .groupby(['constant_speed', 'texture', 'animal_id', 'frame_count'], as_index=False)['instant_speed']
        .mean()
    )

    for (speed, text, animal_id), g in df_mean.groupby(['constant_speed', 'texture', 'animal_id']):
        i = speeds.index(speed)
        j = textures.index(text)

        g = g.sort_values('frame_count')

        ax[i, j].plot(
            g['frame_count'],
            g['instant_speed'],
            alpha=0.3
        )
        ax[i, j].axvline(100, color='black', linestyle='--', linewidth=1)
        
  
    for i, speed in enumerate(speeds):
        if speed == 1.0:
            y_max = 15
            collision_line = 1900
        elif speed == 4.0:
            y_max = 15
            collision_line = 550
        else:
            y_max = 10

        for j in range(len(textures)):
            ax[i, j].set_ylim(0, y_max)
            ax[i, j].set_title(f"{speed} cm/s – {textures[j]}")
            ax[i, j].axvline(collision_line, color='red', linestyle='--', linewidth=1)


    for i in range(len(speeds)):
        ax[i, 0].set_ylabel("instant_speed")
    for j in range(len(textures)):
        ax[-1, j].set_xlabel("frame_count")

    plt.tight_layout()
    return fig, ax


In [ ]:
focal_animal_file

In [ ]:
ready_to_plot1

In [ ]:
speed_over_time_per_ID(ready_to_plot1)

##### 5.7.5: Cumulative heading change by stimulus side
First define the plotting function, then select and plot data for sides 1 and 2.

In [ ]:
def heading_over_time(df):
    df = df.copy()

    textures = sorted(df['texture'].unique())
    speeds   = sorted(df['constant_speed'].unique())

    fig, ax = plt.subplots(len(speeds), len(textures),
                           figsize=(6 * len(textures), 4 * len(speeds)),
                           sharey='row')

    grouped = df.groupby(['constant_speed', 'texture', 'animal_id', 'trial_id'])

    for (speed, text, animal_id, trial_id), group in grouped:
        i = speeds.index(speed)
        j = textures.index(text)

        ax[i, j].plot(group['frame_count'],
                      group['cumulative_heading'],
                      alpha=0.3)

        ax[i, j].axvline(100, color='black', linestyle='--', linewidth=1)
        


    for i, speed in enumerate(speeds):
        if speed == 1.0:
            y_val = 3
            y_max = y_val   
            y_min = -y_val
            collision_line = 1900
            reapearance_line = 2300
        elif speed == 4.0:
            y_max = y_val
            y_min = -y_val
            collision_line = 550    
            reapearance_line = 650 
        else:
            y_max = y_val
            y_min = -y_val

        for j, text in enumerate(textures):
            ax[i, j].set_ylim(y_min, y_max)
            ax[i, j].set_title(f"{speed} cm/s – {text}")
            ax[i, j].axvline(collision_line, color='red', linestyle='--', linewidth=1)
            ax[i, j].axvline(reapearance_line, color='green', linestyle='--', linewidth=1)
            ax[i, j].axhline(0, color='black', linestyle='--', linewidth=1)

    plt.tight_layout()
    return fig, ax



In [ ]:
ready_to_plot_side1 = ready_to_plot1[(ready_to_plot1['side'] == 1.0)]  
                                     #&(ready_to_plot1['animal_id'] == 17.0)]
ready_to_plot_side2 = ready_to_plot1[(ready_to_plot1['side'] == 2.0)]#&(ready_to_plot1['animal_id'] ==17.0)]

In [ ]:
heading_over_time(ready_to_plot_side1)
heading_over_time(ready_to_plot_side2)

##### 5.7.6: Angular velocity by stimulus side
Uses the same side selections as above. Existing axis limits are retained.

In [ ]:


def angle_over_time(df, show_mean=True):
    df = df.copy()

    textures = sorted(df['texture'].unique())
    speeds   = sorted(df['constant_speed'].unique())

    fig, ax = plt.subplots(
        len(speeds), len(textures),
        figsize=(6 * len(textures), 4 * len(speeds)),
        sharey='row'
    )

   
    grouped = df.groupby(['constant_speed', 'texture', 'animal_id', 'trial_id'])

    for (speed, text, animal_id, trial_id), group in grouped:
        i = speeds.index(speed)
        j = textures.index(text)

        ax[i, j].plot(
            group['frame_count'],
            group['instant_angular_velocity'],
            alpha=0.3
        )

        ax[i, j].axvline(100, color='black', linestyle='--', linewidth=1)

    
    if show_mean:
        df_mean = (
            df
            .groupby(['constant_speed', 'texture', 'frame_count'])['instant_angular_velocity']
            .agg(['mean', 'sem'])
            .reset_index()
        )

   
    for i, speed in enumerate(speeds):

        max_val = 2

        if speed == 1.0:
            y_max = max_val
            y_min = -max_val
            collision_line    = 1900
            reapearance_line  = 2300
        elif speed == 4.0:
            y_max = max_val
            y_min = -max_val
            collision_line    = 550
            reapearance_line  = 650
        else:
            y_max = max_val
            y_min = -max_val
            collision_line    = None
            reapearance_line  = None

        for j, text in enumerate(textures):
            ax[i, j].set_ylim(y_min, y_max)
            #ax[i, j].set_xlim(0, df['frame_count'].max())
            ax[i, j].set_title(f"{speed} cm/s – {text}")

            
            ax[i, j].axvline(collision_line,   color='red',   linestyle='--', linewidth=1)
            
            ax[i, j].axvline(reapearance_line, color='green', linestyle='--', linewidth=1)

            ax[i, j].axhline(0, color='black', linestyle='--', linewidth=1)

            if show_mean:
                sub = df_mean[(df_mean['constant_speed'] == speed) &
                              (df_mean['texture'] == text)]
                if not sub.empty:
                    ax[i, j].plot(
                        sub['frame_count'],
                        sub['mean'],
                        color='black',
                        linewidth=2
                    )
                    ax[i, j].fill_between(
                        sub['frame_count'],
                        sub['mean'] - sub['sem'],
                        sub['mean'] + sub['sem'],
                        color='black',
                        alpha=0.15
                    )

    plt.tight_layout()
    return fig, ax



In [ ]:
angle_over_time(ready_to_plot_side1)
angle_over_time(ready_to_plot_side2)

##### Data inspection: Side 1
Full table of the frames previously selected for side 1.

In [ ]:
ready_to_plot_side1.head(220000)
ready_to_plot_side1